In [32]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model

In [33]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train.reshape(-1, 28, 28, 1) / 255.0
x_test  = x_test.reshape(-1, 28, 28, 1) / 255.0

In [34]:
model = models.Sequential([
    layers.Conv2D(2, (5,5), activation='relu', input_shape=(28,28,1)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(4, (5,5), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(input_shape=(28,28,1)),
    layers.Dense(10, activation='sigmoid')
])

In [35]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(x_train, y_train, epochs=30, validation_data=(x_test, y_test))

Epoch 1/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.7002 - loss: 0.9025 - val_accuracy: 0.9258 - val_loss: 0.2412
Epoch 2/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9283 - loss: 0.2395 - val_accuracy: 0.9453 - val_loss: 0.1779
Epoch 3/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9456 - loss: 0.1834 - val_accuracy: 0.9523 - val_loss: 0.1563
Epoch 4/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9540 - loss: 0.1552 - val_accuracy: 0.9629 - val_loss: 0.1221
Epoch 5/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9624 - loss: 0.1250 - val_accuracy: 0.9675 - val_loss: 0.1043
Epoch 6/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9677 - loss: 0.1075 - val_accuracy: 0.9699 - val_loss: 0.0909
Epoch 7/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9685 - loss: 0.0997 - val_accuracy: 0.9726 - val_loss: 0.0859
Epoch 8/30
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9719 - loss: 0.0956 - 

In [36]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 24, 24, 2)      │            52 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 12, 12, 2)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 8, 8, 4)        │           204 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 4, 4, 4)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,720 (10.63 KB)

 Trainable params: 906 (3.54 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,814 (7.09 KB)

In [37]:
from tensorflow.keras.models import Model
encoder = Model(inputs=model.layers[0].input, outputs=model.layers[-2].output)  # -2 to exclude the last layer
encoder

<Functional name=functional_15, built=True>

In [38]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test accuracy:", test_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9755 - loss: 0.0800
Test accuracy: 0.9804999828338623


In [39]:
import numpy as np

y_pred = model.predict(x_test)
y_pred_labels = np.argmax(y_pred, axis=1)  # OK

y_true = y_test  # already class indices

manual_acc = np.mean(y_pred_labels == y_true)
print(manual_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
0.9805


In [40]:
model.save("model.keras")

layer = model.get_layer("dense_1")


In [41]:
import numpy as np
from tensorflow.keras.models import load_model
import struct

In [42]:
MODEL_PATH = "model.keras"
LAYER_NAME = "dense_1"

# Output file names
OUT_FLOAT32_W = "dense_weights_float32_bin.txt"
OUT_FLOAT32_B = "dense_biases_float32_bin.txt"

OUT_FLOAT16_W = "dense_weights_float16_bin.txt"
OUT_FLOAT16_B = "dense_biases_float16_bin.txt"

OUT_FLOAT64_W = "dense_weights_float64_bin.txt"
OUT_FLOAT64_B = "dense_biases_float64_bin.txt"

OUT_FIXED16_BIN_W = "dense_weights_fixed16_bin_1d.txt"
OUT_FIXED16_BIN_B = "dense_biases_fixed16_bin_1d.txt"

OUT_FIXED8_BIN_W = "dense_weights_fixed8_bin.txt"
OUT_FIXED8_BIN_B = "dense_biases_fixed8_bin.txt"


# ---------------------------------------------------
# Helper functions
# ---------------------------------------------------

def float32_to_bin(x):
    """Convert Python float to IEEE754 float32 (binary string of 0/1)."""
    # Float → 4 bytes (IEEE754 float32)
    raw = struct.pack('!f', x)
    # Bytes → 32-bit integer
    uint32 = struct.unpack('!I', raw)[0]
    # Integer → 32-bit binary string
    return format(uint32, '032b')

def float16_to_bin(x):
    """Convert float16 to IEEE754 binary string."""
    # Convert float16 → uint16 bit pattern
    u = np.frombuffer(np.float16(x).tobytes(), dtype=np.uint16)[0]
    return format(u, '016b')

def float64_to_bin(x):
    """Convert float64 to IEEE754 binary string."""
    return format(struct.unpack('!Q', struct.pack('!d', x))[0], '064b')

def to_binary_signed(x, bits):
    """Convert int8/int16/int32 to two's complement binary safely using NumPy."""
    # Force NumPy dtype with correct width
    if bits == 8:
        x = np.int8(x)
        u = x.view(np.uint8)
    elif bits == 16:
        x = np.int16(x)
        u = x.view(np.uint16)
    elif bits == 32:
        x = np.int32(x)
        u = x.view(np.uint32)
    else:
        raise ValueError("Unsupported bit width")

    return format(u, f"0{bits}b")

def float64_to_bin(x):
    """Convert float64 to IEEE754 64-bit binary string."""
    # Pack float64 into bytes, then unpack as uint64
    u = struct.unpack('!Q', struct.pack('!d', x))[0]
    return format(u, '064b')

def save_list_binary(values, filename, fn):
    """Save list converted by function fn (float or int)."""
    with open(filename, "w") as f:
        for v in values:
            f.write(fn(v) + "\n")


# ---------------------------------------------------
# Load model
# ---------------------------------------------------

model = load_model(MODEL_PATH)
layer = model.get_layer(LAYER_NAME)

weights, biases = layer.get_weights()
weights_flat = weights.flatten()
biases_flat = biases.flatten()

# ---------------------------------------------------
# Save FLOAT32 → IEEE754 binary
# ---------------------------------------------------

save_list_binary(weights_flat, OUT_FLOAT32_W, float32_to_bin)
save_list_binary(biases_flat,  OUT_FLOAT32_B, float32_to_bin)

# ---------------------------------------------------
# Save FLOAT16 → IEEE754 binary
# ---------------------------------------------------

save_list_binary(weights_flat, OUT_FLOAT16_W, float16_to_bin)
save_list_binary(biases_flat,  OUT_FLOAT16_B, float16_to_bin)

# ---------------------------------------------------
# Save FLOAT64 → IEEE754 binary
# ---------------------------------------------------

save_list_binary(weights_flat, OUT_FLOAT64_W, float64_to_bin)
save_list_binary(biases_flat,  OUT_FLOAT64_B, float64_to_bin)

# ---------------------------------------------------
# FIXED-POINT CONVERSION (with clamping)
# ---------------------------------------------------

SCALE16 = 2**8   # Q8.8
SCALE8  = 2**4   # Q4.4

# Compute scaled values (float first)
wf16 = np.round(weights_flat * SCALE16)
bf16 = np.round(biases_flat  * SCALE16)

wf8 = np.round(weights_flat * SCALE8)
bf8 = np.round(biases_flat  * SCALE8)

# Clip to int16 range
wf16 = np.clip(wf16, -32768, 32767).astype(np.int16)
bf16 = np.clip(bf16, -32768, 32767).astype(np.int16)

# Clip to int8 range
wf8 = np.clip(wf8, -128, 127).astype(np.int8)
bf8 = np.clip(bf8, -128, 127).astype(np.int8)



# ---------------------------------------------------
# Save FIXED16 / FIXED8 integer binary
# ---------------------------------------------------

save_list_binary(wf16, OUT_FIXED16_BIN_W, lambda x: to_binary_signed(np.int16(x), 16))
save_list_binary(bf16,  OUT_FIXED16_BIN_B, lambda x: to_binary_signed(np.int16(x), 16))

save_list_binary(wf8, OUT_FIXED8_BIN_W, lambda x: to_binary_signed(np.int8(x), 8))
save_list_binary(bf8,  OUT_FIXED8_BIN_B, lambda x: to_binary_signed(np.int8(x), 8))

print("All files generated successfully!")

All files generated successfully!


In [43]:
from google.colab import files
files.download("dense_weights_fixed8_bin.txt")
files.download("dense_biases_fixed16_bin_1d.txt")
files.download("dense_weights_float32_bin.txt")
files.download("dense_biases_float32_bin.txt")
files.download("dense_weights_float16_bin.txt")
files.download("dense_biases_float16_bin.txt")
files.download("dense_biases_fixed8_bin.txt")
files.download("dense_biases_float64_bin.txt")
files.download("dense_weights_float64_bin.txt")
files.download("dense_weights_fixed16_bin_1d.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

extracting the inputs file

In [63]:
# Load the saved model
model = load_model("model.keras")
N = 10000
# Take one sample input to build the model
X_sample = x_test[:N]  # shape (1,28,28,1)
_ = model.predict(X_sample)  # this builds the model and defines input/output tensors

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [64]:
pred = model.predict(X_sample)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [65]:
pred_digits = pred.argmax(axis=1)

In [66]:
np.savetxt("colab_predictions.txt", pred_digits, fmt="%d")

In [55]:
with open("colab_predictions.txt", "w") as f:
    for d in pred_digits:
        f.write(f"{d:04b}\n")

In [ ]:
files.download("colab_predictions.txt")


In [67]:
_ = model(X_sample)

In [68]:
for i, layer in enumerate(model.layers):
    try:
        print(i, layer.name, layer.output.shape)
    except:
        print(i, layer.name)

0 conv2d_2 (None, 24, 24, 2)
1 max_pooling2d_2 (None, 12, 12, 2)
2 conv2d_3 (None, 8, 8, 4)
3 max_pooling2d_3 (None, 4, 4, 4)
4 flatten_1 (None, 64)
5 dense_1 (None, 10)


In [69]:
flatten_layer_index = 4  # Flatten layer
x = X_sample
for i in range(flatten_layer_index + 1):  # include flatten layer
    x = model.layers[i](x)

flatten_output_flat = x.numpy().flatten()  # shape (64,)
print(flatten_output_flat.shape)

(640000,)


In [70]:
# Float32
save_list_binary(flatten_output_flat, "flatten_output_10000_images_float32_bin.txt", float32_to_bin)

# Float64
save_list_binary(flatten_output_flat, "flatten_output_10000_images_float64_bin.txt", float64_to_bin)

# Float16

save_list_binary(flatten_output_flat, "flatten_output_10000_images_float16_bin.txt", float16_to_bin)


# Fixed16 Q8.8
SCALE16 = 2**8
#flatten_fixed16 = np.clip(np.round(flatten_output_flat * SCALE16), -32768, 32767).astype(np.int16)
flatten_fixed16 = np.round(flatten_output_flat * SCALE16)
flatten_fixed16 = np.clip(flatten_fixed16, -32768, 32767).astype(np.int16)
#save_list_binary(flatten_fixed16, "flatten_output_fixed16_bin.txt", lambda x: to_binary_signed(x, 16))
save_list_binary(flatten_fixed16, "flatten_output_10000_images_fixed16_bin.txt", lambda x: to_binary_signed(np.int16(x), 16))

# Fixed8 Q4.4
SCALE8 = 2**4
#flatten_fixed8 = np.clip(np.round(flatten_output_flat * SCALE8), -128, 127).astype(np.int8)
flatten_fixed8 = np.round(flatten_output_flat * SCALE8)
flatten_fixed8 = np.clip(flatten_fixed8, -128, 127).astype(np.int8)
save_list_binary(flatten_fixed8, "flatten_output_10000_images_fixed8_bin.txt", lambda x: to_binary_signed(np.int8(x), 8))




In [ ]:
from google.colab import files
files.download("flatten_output_10000_images_float64_bin.txt")
files.download("flatten_output_10000_images_float32_bin.txt")
files.download("flatten_output_10000_images_float16_bin.txt")
files.download("flatten_output_10000_images_fixed16_bin.txt")
files.download("flatten_output_10000_images_fixed8_bin.txt")
